In [2]:
import pandas as pd
import plotly.express as px
import numpy as np
import re

# 1. Nomes EXATOS dos arquivos
arquivos = [
    "results_30.csv",
]

def extrair_valor_positivo(valor):
    """
    Converte strings de arrays '[neg pos]' para o float da classe positiva.
    Se já for número, apenas retorna o número.
    """
    if isinstance(valor, str):
        # Remove colchetes e divide por espaços ou vírgulas
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", valor)
        if len(numeros) > 1:
            return float(numeros[1]) # Pega a classe positiva (índice 1)
        elif len(numeros) == 1:
            return float(numeros[0])
    return valor

lista_df = []
for f in arquivos:
    try:
        temp_df = pd.read_csv(f)
        temp_df.columns = [c.lower() for c in temp_df.columns]
        
        # Aplicar a limpeza na coluna 'erro' e 'prev_pred' antes de empilhar
        if 'erro' in temp_df.columns:
            temp_df['erro'] = temp_df['erro'].apply(extrair_valor_positivo)
        
        lista_df.append(temp_df)
        print(f"✅ {f} carregado e limpo!")
    except Exception as e:
        print(f"❌ Erro ao ler {f}: {e}")

# Combina tudo
df = pd.concat(lista_df, ignore_index=True)

# 2. TRATAMENTO FINAL
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["erro", "modelo"])

# 3. ORDENAÇÃO
order = (
    df.groupby("modelo")["erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# 4. GRÁFICO
fig = px.box(
    df,
    x="modelo",
    y="erro",
    category_orders={"modelo": order},
    points="all",
    color="modelo",
    color_discrete_sequence=px.colors.qualitative.Dark24
)

fig.update_layout(
    title="Comparação de Erro entre Modelos (Saídas Normalizadas)",
    xaxis_title="Modelo",
    yaxis_title="Erro absoluto (Classe Positiva)",
    template="simple_white",
    width=1200, 
    height=600,
    showlegend=False # Oculto para limpar o visual, já está no eixo X
)

fig.update_xaxes(tickangle=45)
fig.show()

print("\nModelos processados com sucesso:")
print(df["modelo"].unique())

/tmp/ipykernel_3214503/719339527.py:28: DtypeWarning:

Columns (2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.



✅ results_30.csv carregado e limpo!



Modelos processados com sucesso:
['DyS_Topsoe' 'QuaDapt_DyS' 'E_tsfresh' 'MiniRocket' 'F_catch22']
